# 02. 관계형 데이터베이스(RDB) 입문 — 금속 물성표를 SQL로 다루기

논문을 읽다 보면 **물성표(Table)** 를 자주 만납니다. "비저항이 Cu보다 낮은 금속은?", "ρ·λ 가 가장 작은 후보 3개는?" 같은 질문은
본문을 읽는 것보다 **표에서 숫자를 비교**하는 편이 훨씬 정확합니다.
이번 장에서는 표 데이터를 다루는 가장 오래되고 믿음직한 도구인 **관계형 데이터베이스(Relational Database, RDB)** 와
그 질의 언어 **SQL** 을 배우고, 마지막에는 LLM이 한국어 질문을 SQL로 바꿔 주는 **Text2SQL** 을 직접 만들어 봅니다.

### 학습 목표
- 테이블·행·열·기본키(Primary Key)·외래키(Foreign Key) 개념을 물성표에 빗대어 이해합니다.
- 파이썬 내장 `sqlite3` 로 테이블을 만들고 `INSERT`, `SELECT`, `WHERE`, `ORDER BY`, `JOIN`, `GROUP BY` 를 직접 써 봅니다.
- 논문 표에서 옮겨 적은 실제 데이터(Sung et al. 2026 Table 1·2, Kim et al. 2016 Table 1)로 DB를 만들고 질의합니다.
- LLM에게 SQL을 쓰게 하는 Text2SQL 을 직접 구현하고, **SELECT만 허용하는 안전장치**가 왜 필요한지 이해합니다.
- RDB의 강점(정확한 수치·조건 검색)과 한계(비정형 텍스트·"의미" 검색 불가)를 구분합니다.

## 환경설정

In [ ]:
# [Colab 전용] 저장소를 내려받고 패키지·Ollama·모델을 설치합니다. (로컬 Jupyter에서는 자동으로 건너뜁니다)
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/skku_rag_tutorials"):
        !git clone -q https://github.com/JDeun/skku_rag_tutorials.git /content/skku_rag_tutorials
    %cd /content/skku_rag_tutorials
    !bash scripts/colab_setup.sh

In [ ]:
import warnings; warnings.filterwarnings("ignore")   # 불필요한 경고 숨기기
import sqlite3                                        # 파이썬에 기본 내장된 RDB 엔진
import pandas as pd                                   # 표를 보기 좋게 출력하는 도구
from skku_rag import get_llm, LLM_MODEL, EMBED_MODEL, PROJECT_ROOT

print("LLM:", LLM_MODEL, "| 임베딩:", EMBED_MODEL)

## 1. 데이터를 저장하는 여러 방법

연구실에서 데이터를 기록하는 방식을 떠올려 봅시다.

| 연구실에서의 기록 | 컴퓨터에서의 저장 방식 | 잘하는 일 | 못하는 일 |
|---|---|---|---|
| 실험노트(자유롭게 쓴 글) | 텍스트 파일, PDF | 맥락·이유·해석을 풍부하게 기록 | "비저항 < 5 인 시료만" 같은 조건 검색 |
| 엑셀 물성표 | CSV / 엑셀 | 숫자 비교, 정렬 | 여러 표를 연결, 여러 사람이 동시에 안전하게 수정 |
| **물성 데이터베이스** | **관계형 DB (SQLite, MySQL, PostgreSQL)** | 여러 표를 **키로 연결**, 정확한 조건 검색, 대용량 | 문장의 "의미"로 찾기 |
| 시료 보관함을 "비슷한 것끼리" 정리 | 벡터 DB (03장) | 의미가 비슷한 글 찾기 | 정확한 수치 조건 |
| 재료–공정–물성 관계도 | 그래프 DB (뒤의 장) | 관계를 따라가는 추론 | 대량 수치 집계 |

RAG(검색 증강 생성)는 결국 **"질문에 맞는 자료를 어디서, 어떻게 꺼내 오느냐"** 의 문제입니다.
질문이 숫자·조건이면 RDB가, 의미·설명이면 벡터DB가, 관계면 그래프DB가 유리합니다. 이번 장은 RDB입니다.

## 2. 테이블 · 행 · 열 · 기본키 · 외래키

RDB는 **엑셀 시트와 비슷한 표(테이블, Table)** 여러 개로 이루어집니다.

```
 papers (논문 테이블)                      metals (물성 테이블)
+----------+---------------------+        +----------+----------+-------+----------+
| paper_id | title               |        | metal_id | material | rho   | paper_id |
+----------+---------------------+        +----------+----------+-------+----------+
| kim2016  | Interplay ...       |        | 1        | Cu       | 1.678 | sung2026 |--+
| sung2026 | Recent progress ... |<-------| 2        | Al       | 2.650 | sung2026 |--+
+----------+---------------------+        +----------+----------+-------+----------+
  ^ 기본키(PK)                               ^ 기본키(PK)                ^ 외래키(FK) → papers.paper_id
```

| 용어 | 뜻 | 물성표 비유 |
|---|---|---|
| 테이블(Table) | 같은 종류의 데이터를 모은 표 | 엑셀 시트 하나 |
| 행(Row, 레코드) | 데이터 한 건 | 시료 한 개의 측정 결과 한 줄 |
| 열(Column) | 속성 하나, **자료형(type)이 정해짐** | "비저항(μΩ·cm)" 열 — 숫자만 들어감 |
| 기본키(Primary Key, PK) | 각 행을 유일하게 구별하는 값 | 시료 번호 (중복 불가) |
| 외래키(Foreign Key, FK) | 다른 테이블의 기본키를 가리키는 값 | "이 데이터는 어느 논문 표에서 왔나" 를 적은 칸 |

> 핵심: 논문 제목을 물성 행마다 반복해서 쓰지 않고, **`paper_id` 하나만 적어 두고 필요할 때 연결(JOIN)** 합니다.
> 그래서 이름이 **관계형(Relational)** 데이터베이스입니다.

## 3. sqlite3 로 직접 테이블 만들기

SQLite 는 **파일 하나(또는 메모리)** 로 동작하는 가벼운 RDB입니다. 별도 서버 설치가 필요 없고 파이썬에 기본으로 들어 있습니다.
먼저 연습용으로 메모리(`:memory:`)에 DB를 만들고, **Sung et al. 2026 리뷰의 Table 1** 에서 몇 개 금속을 옮겨 봅니다.

- `ρ0` (resistivity, μΩ·cm): 벌크 비저항
- `λ` (EMFP, nm): 전자 평균 자유 행로 — 전자가 산란 없이 달리는 평균 거리
- `ρ0·λ` (10⁻¹⁶ Ω·m²): 미세 배선 성능지수(FOM). **작을수록 선폭이 줄어도 저항이 덜 커짐**

In [ ]:
conn = sqlite3.connect(":memory:")   # 메모리 위에 임시 DB 생성 (파일로 저장되지 않음)

# CREATE TABLE: 표의 '틀(열 이름과 자료형)'을 먼저 정합니다.
conn.execute("""
CREATE TABLE toy_metals (
    name        TEXT PRIMARY KEY,   -- 금속 이름 (기본키: 중복 불가)
    structure   TEXT,               -- 결정 구조
    rho         REAL,               -- 비저항 ρ0 (μΩ·cm)
    emfp        REAL,               -- 전자 평균 자유 행로 λ (nm)
    rho_lambda  REAL                -- ρ0·λ (1e-16 Ω·m²)
)""")
print("테이블 생성 완료")

In [ ]:
# INSERT: 행(데이터)을 넣습니다. 값 자리에는 ? 를 쓰고 값은 따로 넘깁니다(안전한 방식).
rows = [  # Sung et al. 2026, Table 1 에서 옮긴 값
    ("Cu", "fcc", 1.678, 39.9, 6.70),
    ("Al", "fcc", 2.650, 18.9, 5.01),
    ("Rh", "fcc", 4.7,   6.88, 3.23),
    ("Ir", "fcc", 5.2,   7.09, 3.69),
    ("Mo", "bcc", 5.34,  11.2, 5.99),
    ("Nb", "bcc", 14.75, 2.36, 3.56),
]
conn.executemany("INSERT INTO toy_metals VALUES (?, ?, ?, ?, ?)", rows)
conn.commit()  # 변경 사항 확정
print(conn.execute("SELECT COUNT(*) FROM toy_metals").fetchone()[0], "행 입력 완료")

기본키는 **중복을 막아 줍니다.** 같은 `Cu` 를 한 번 더 넣으면 어떻게 될까요? (실험노트에 같은 시료 번호를 두 번 쓰는 실수를 DB가 막아 주는 셈입니다.)

In [ ]:
try:
    conn.execute("INSERT INTO toy_metals VALUES ('Cu', 'fcc', 1.7, 40, 6.8)")
except sqlite3.IntegrityError as e:
    print("입력 거부됨 →", e)   # UNIQUE constraint failed: 기본키 중복

### 3-1. SELECT · WHERE · ORDER BY

SQL 문장은 영어 문장처럼 읽힙니다.

```
SELECT  보고 싶은 열        -- 무엇을
FROM    테이블              -- 어디에서
WHERE   조건                -- 어떤 행만
ORDER BY 정렬 기준 [DESC]   -- 어떤 순서로
LIMIT   개수                -- 몇 개만
```

In [ ]:
# 전체 보기: * 는 '모든 열'
for row in conn.execute("SELECT * FROM toy_metals"):
    print(row)

In [ ]:
# 질문: "비저항이 Cu보다 낮은 금속은?"
# 하위 질의(subquery)로 Cu의 비저항을 먼저 구해 비교합니다.
sql = """
SELECT name, rho FROM toy_metals
WHERE rho < (SELECT rho FROM toy_metals WHERE name = 'Cu')
"""
print(conn.execute(sql).fetchall())

결과가 **빈 목록** 입니다. 벌크 비저항만 보면 Cu가 가장 낮기 때문에 지금까지 Cu가 배선 재료로 쓰여 왔습니다.
그렇다면 성능지수 **ρ0·λ** 로 비교하면 어떨까요?

In [ ]:
# 질문: "ρ·λ가 가장 작은 후보 3개는?"  → 작은 순(ASC) 정렬 후 3개만
sql = "SELECT name, rho, emfp, rho_lambda FROM toy_metals ORDER BY rho_lambda ASC LIMIT 3"
for row in conn.execute(sql):
    print(row)

Rh, Nb, Ir 처럼 **벌크 비저항은 Cu보다 높지만 λ가 짧아 ρ0·λ 가 작은** 금속들이 나옵니다.
리뷰 논문이 "벌크 비저항만으로는 나노 스케일 배선 성능을 설명할 수 없다"고 말하는 이유를 SQL 두 줄로 확인한 셈입니다.

### 3-2. pandas 로 결과를 표로 보기
`pd.read_sql_query` 를 쓰면 SQL 결과가 바로 보기 좋은 표(DataFrame)로 나옵니다.

In [ ]:
pd.read_sql_query("SELECT * FROM toy_metals WHERE structure = 'fcc' ORDER BY rho", conn)

### 3-3. 집계(GROUP BY): 결정 구조별 평균
`GROUP BY` 는 "같은 값끼리 묶어서 계산" 합니다. 엑셀의 피벗 테이블과 같은 역할입니다.

In [ ]:
sql = """
SELECT structure, COUNT(*) AS n, ROUND(AVG(rho), 2) AS avg_rho, MIN(rho_lambda) AS min_rho_lambda
FROM toy_metals
GROUP BY structure
"""
pd.read_sql_query(sql, conn)

In [ ]:
conn.close()   # 연습용 DB 닫기 (메모리 DB라 닫으면 사라집니다)

## 4. 논문 표 전체로 DB 만들기 — `build_sqlite`

이번에는 두 논문의 표를 **사람이 직접 옮겨 적은 CSV** 로부터 DB 파일을 만듭니다. (PDF 표 자동 추출은 오류가 잦아, 수치는 논문 표를 보고 그대로 옮겼습니다.)

| CSV 파일 | 테이블 | 내용 |
|---|---|---|
| `paper_data/papers.csv` | `papers` | 논문 2편의 제목·저자·저널·연도·DOI |
| `paper_data/metal_properties.csv` | `metals` | Sung 2026 Table 1(원소 금속 11종) + Table 2(화합물·산화물 16종, 계산값) |
| `paper_data/alloy_films.csv` | `alloy_films` | Kim 2016 Table 1: Cu-Nb, Cu-Fe 박막의 조성·두께·결정립 크기 |

`skku_rag/sql_store.py` 의 `build_sqlite()` 는 방금 3장에서 손으로 한 일(CREATE TABLE → INSERT)을 CSV 3개에 대해 반복할 뿐입니다.

In [ ]:
from skku_rag.sql_store import build_sqlite, SQL_SCHEMA_DESCRIPTION

db_path = build_sqlite()   # db/papers.sqlite 생성 (이미 있으면 그대로 사용)
print("DB 파일:", db_path.relative_to(PROJECT_ROOT))
conn = sqlite3.connect(db_path)

In [ ]:
# sqlite_master: DB가 스스로 기록해 둔 '테이블 목록'
pd.read_sql_query("SELECT name FROM sqlite_master WHERE type = 'table' AND name NOT LIKE 'sqlite_%'", conn)

In [ ]:
# PRAGMA table_info: 테이블의 열 이름과 자료형, 기본키(pk) 여부 확인
pd.read_sql_query("PRAGMA table_info(metals)", conn)[["name", "type", "pk"]]

In [ ]:
pd.read_sql_query("SELECT material, category, crystal_structure, resistivity_uohm_cm, emfp_nm, "
                  "rho_lambda_1e16_ohm_m2, source FROM metals LIMIT 14", conn)

빈칸은 `None`(SQL에서는 **NULL**)입니다. 예를 들어 Table 2에는 결정 구조 열이 없으므로 비워 두었습니다.
**없는 값을 지어내지 않는 것** 은 연구 데이터와 마찬가지로 DB에서도 가장 중요한 원칙입니다.

### 4-1. 실제 데이터로 질의하기

In [ ]:
# 원소 금속(Table 1) 중 ρ·λ 가 Cu보다 작은 것 — 크기 효과에 강한 후보
sql = """
SELECT material, crystal_structure, resistivity_uohm_cm AS rho, emfp_nm, rho_lambda_1e16_ohm_m2 AS rho_lambda
FROM metals
WHERE source = 'sung2026 Table 1'
  AND rho_lambda_1e16_ohm_m2 < (SELECT rho_lambda_1e16_ohm_m2 FROM metals
                                WHERE material = 'Cu' AND source = 'sung2026 Table 1')
ORDER BY rho_lambda
"""
pd.read_sql_query(sql, conn)

In [ ]:
# Table 2(계산값): 응집 에너지(전자이동 내성의 지표)가 6 eV/atom 이상이면서 ρ·λ 가 5 이하인 화합물
sql = """
SELECT material, rho_lambda_1e16_ohm_m2 AS rho_lambda, cohesive_energy_ev_atom AS e_coh
FROM metals
WHERE category != 'element' AND cohesive_energy_ev_atom >= 6 AND rho_lambda_1e16_ohm_m2 <= 5
ORDER BY rho_lambda
"""
pd.read_sql_query(sql, conn)

### 4-2. JOIN: 두 테이블 연결하기
`metals.paper_id` (외래키)와 `papers.paper_id` (기본키)가 같은 행끼리 이어 붙이면, 물성값 옆에 **출처 논문 정보**를 붙일 수 있습니다.

In [ ]:
sql = """
SELECT m.material, m.rho_lambda_1e16_ohm_m2 AS rho_lambda, p.journal, p.year, m.source
FROM metals AS m
JOIN papers AS p ON m.paper_id = p.paper_id
ORDER BY rho_lambda
LIMIT 5
"""
pd.read_sql_query(sql, conn)

### 4-3. GROUP BY: Kim 2016 합금 박막 요약
Kim et al. (2016)은 Cu에 Nb, Fe를 소량 넣었을 때 결정립이 오히려 커진다고 보고했습니다. 첨가 원소별로 묶어 봅니다.

In [ ]:
sql = """
SELECT solute, COUNT(*) AS n_films,
       MIN(solute_at_pct) AS min_at_pct, MAX(solute_at_pct) AS max_at_pct,
       MAX(grain_size_nm) AS max_grain_nm
FROM alloy_films
GROUP BY solute
"""
pd.read_sql_query(sql, conn)

In [ ]:
# 결정립이 가장 큰 시료 3개 (평균 결정립 크기 기준)
pd.read_sql_query("SELECT solute, solute_at_pct, grain_size_nm, grain_size_err_nm "
                  "FROM alloy_films ORDER BY grain_size_nm DESC LIMIT 3", conn)

순수 Cu(10.1 nm)보다 **Fe 0.73 at.%(14.5 nm), Nb 1.14 at.%(13.6 nm)** 시료의 결정립이 더 큽니다 — 논문 본문의 주장과 일치합니다.

## 5. Text2SQL 맛보기 — LLM에게 SQL을 쓰게 하기

학생이 SQL을 몰라도 한국어로 물으면 답을 얻을 수 있을까요? 아이디어는 단순합니다.

```
 한국어 질문 ──► [LLM] ──► SQL 문 ──► [SQLite 실행] ──► 결과 표
                  ▲
       테이블 설명서(스키마) 함께 제공
```

LLM은 우리 DB의 테이블 이름·열 이름·단위를 모르므로 **설명서(스키마 설명)** 를 프롬프트에 함께 넣어야 합니다.
`SQL_SCHEMA_DESCRIPTION` 이 바로 그 설명서입니다.

In [ ]:
print(SQL_SCHEMA_DESCRIPTION[:1200], "...")

### 5-1. 프롬프트 만들기

In [ ]:
PROMPT_TEMPLATE = """You are an expert SQLite assistant.
Write ONE SQLite SELECT query that answers the question, using only the tables below.

{schema}

Rules:
- Output only the SQL inside a ```sql code block. No explanation.
- Use only SELECT (read-only). Use exact column names from the schema.

Question (Korean): {question}
"""
llm = get_llm()   # 기본 LLM (생각 과정 출력 꺼짐)

### 5-2. 안전장치: SELECT 만 허용하기

LLM이 만든 SQL을 **검사 없이 그대로 실행하면 위험** 합니다. 모델이 실수(또는 악의적인 질문에 속아서)로
`DROP TABLE metals` (테이블 삭제)나 `DELETE FROM ...` 을 만들면 데이터가 사라집니다.
실험실에서 신입에게 장비를 맡길 때 "측정만 하고 설정은 바꾸지 마" 라고 하는 것과 같습니다. 그래서 두 겹으로 막습니다.

1. **문장 검사**: `SELECT`(또는 `WITH`)로 시작하는 문장 하나만 허용, 수정 관련 키워드가 있으면 거부
2. **읽기 전용 연결**: DB 파일을 `mode=ro`(read-only)로 열어, 혹시 검사를 통과해도 수정이 불가능하게

In [ ]:
import re

FORBIDDEN = re.compile(r"\b(INSERT|UPDATE|DELETE|DROP|ALTER|CREATE|REPLACE|ATTACH|DETACH|PRAGMA|VACUUM)\b", re.I)

def extract_sql(text: str) -> str:
    """LLM 답변에서 ```sql ... ``` 안의 SQL만 꺼냅니다(코드 블록이 없으면 전체 사용)."""
    match = re.search(r"```(?:sql)?\s*(.*?)```", text, re.S | re.I)
    sql = match.group(1) if match else text
    return sql.strip().rstrip(";").strip()

def check_safe(sql: str) -> None:
    """읽기 전용 SELECT 한 문장인지 검사하고, 아니면 오류를 냅니다."""
    if not re.match(r"^(SELECT|WITH)\b", sql, re.I):
        raise ValueError("SELECT 로 시작하는 문장만 허용합니다.")
    if ";" in sql or FORBIDDEN.search(sql):
        raise ValueError("여러 문장이나 데이터 수정 명령은 허용하지 않습니다.")

In [ ]:
# 안전장치 시험: 위험한 SQL은 실행 전에 거부됩니다.
for bad in ["DROP TABLE metals", "SELECT * FROM metals; DELETE FROM metals", "SELECT material FROM metals"]:
    try:
        check_safe(bad)
        print("통과  :", bad)
    except ValueError as e:
        print("거부됨:", bad, "→", e)

In [ ]:
from contextlib import closing
from pathlib import Path

def run_readonly(sql: str) -> pd.DataFrame:
    """DB를 읽기 전용(mode=ro)으로 열어 SQL을 실행합니다."""
    uri = Path(db_path).resolve().as_uri() + "?mode=ro"   # file:///.../papers.sqlite?mode=ro (Windows 경로도 안전)
    with closing(sqlite3.connect(uri, uri=True)) as ro_conn:   # closing: 블록이 끝나면 연결을 실제로 닫음
        return pd.read_sql_query(sql, ro_conn)

### 5-3. 질문 → SQL → 실행 (간단 재시도 포함)

작은 모델은 열 이름을 틀리거나 형식을 어기기도 합니다. 그래서 **실행 오류 메시지를 다시 LLM에게 보여 주고 고치게** 합니다(최대 2번 시도).
사람이 SQL 오류를 보고 고치는 과정과 같습니다.

In [ ]:
def text2sql(question: str, max_tries: int = 2):
    """한국어 질문을 SQL로 바꿔 실행하고 (SQL, 결과표)를 돌려줍니다."""
    prompt = PROMPT_TEMPLATE.format(schema=SQL_SCHEMA_DESCRIPTION, question=question)
    sql = ""
    for attempt in range(1, max_tries + 1):
        sql = extract_sql(llm.invoke(prompt).content)
        try:
            check_safe(sql)
            return sql, run_readonly(sql)
        except Exception as e:   # 안전 검사 실패 또는 SQL 실행 오류
            print(f"[시도 {attempt}] 실패: {e}")
            prompt += f"\n\nYour previous SQL:\n{sql}\nfailed with error: {e}\nFix it and output only the SQL."
    return sql, None

In [ ]:
sql, result = text2sql("ρ·λ 값이 가장 작은 원소 금속 3개를 알려줘")
print(sql)
result

생성된 SQL이 맞을까요? 결과는 실행할 때마다, 모델마다 다르지만 소형 모델에서는 이런 일이 자주 생깁니다.
- 첫 시도에서 **없는 열 이름을 지어내** 실패하고, 오류 메시지를 보고 재시도해 **실행은 되는** SQL을 만듭니다.
- 그런데 이미 ρ0·λ 값인 `rho_lambda_1e16_ohm_m2` 에 ρ0를 **한 번 더 곱하는** 식으로 계산을 잘못하거나, `source` 조건을 빠뜨려 Table 1·2 의 Cu가 **두 번** 나오기도 합니다.

여러분의 결과에서 생성된 SQL과 순위를 직접 확인해 보세요. **오류 없이 실행된다고 답이 맞는 것은 아닙니다.**

사람이 직접 쓴 **정답 SQL** 과 비교해 봅니다. (원소 금속은 Table 1 에서만 골라야 Cu가 두 번 나오지 않습니다.)

In [ ]:
answer_sql = """
SELECT material, rho_lambda_1e16_ohm_m2 FROM metals
WHERE source = 'sung2026 Table 1' ORDER BY rho_lambda_1e16_ohm_m2 LIMIT 3
"""
run_readonly(answer_sql)

In [ ]:
sql, result = text2sql("hcp 구조인 금속들의 평균 비저항은 얼마야?")
print(sql)
result

In [ ]:
sql, result = text2sql("Fe를 첨가한 박막 중 결정립이 가장 큰 시료의 Fe 농도와 결정립 크기는?")
print(sql)
result

**결과 해설**: 생성된 SQL을 꼭 직접 읽어 보세요. 작은 4B 모델은 다음과 같은 실수를 자주 합니다.

- 열 이름을 비슷하게 지어냄 (예: `resistivity` → 실제는 `resistivity_uohm_cm`) → 실행 오류 → 재시도로 고쳐지는 경우가 많음
- `source` 조건을 빠뜨려 Table 1(원소)과 Table 2(화합물)를 섞음 → **오류 없이 실행되지만 의도와 다른 답** (가장 위험!)
- `ORDER BY` 방향(ASC/DESC)을 반대로 씀

실행 오류는 재시도로 잡을 수 있지만, **"그럴듯하게 틀린 SQL"** 은 사람이 확인해야 합니다. 그래서 Text2SQL 결과에는
항상 생성된 SQL을 함께 보여 주는 것이 좋습니다. 스키마 설명을 더 자세히 쓰거나 예시 질문–SQL 쌍(few-shot)을 넣으면 정확도가 올라갑니다.

## 6. RDB의 강점과 한계

**강점**: 숫자·조건·정렬·집계는 RDB가 가장 정확합니다. "ρ·λ < 4 인 금속" 은 100% 정확한 답을 줍니다 — LLM이 기억으로 대답하면 숫자를 틀리기 쉽습니다.

**한계**: 그런데 이런 질문은 어떨까요?

- "Cu 배선의 비저항이 선폭이 줄면 왜 커지나요?"
- "Nb를 넣었더니 결정립이 커진 이유는?"

답은 **논문 본문(문장)** 에 있고, 표에는 없습니다. 본문을 DB에 넣는다고 해도 SQL은 **글자가 똑같은지** 만 비교할 수 있습니다.

In [ ]:
# note 열에서 'scattering' 이라는 글자를 찾아봄 → 표에는 '이유'가 적혀 있지 않음
print(pd.read_sql_query("SELECT material, note FROM metals WHERE note LIKE '%scattering%'", conn))
# 한국어 '산란' 으로 찾으면 영어 데이터와 글자가 달라 당연히 못 찾음
print(pd.read_sql_query("SELECT material FROM metals WHERE note LIKE '%산란%'", conn))

`LIKE '%...%'` 는 **철자 일치** 검색일 뿐이라 "산란" 과 "scattering" 이 같은 뜻이라는 것을 모릅니다.
**뜻(의미)으로 찾는 검색** 을 하려면 문장을 숫자 벡터로 바꾸는 **임베딩과 벡터DB** 가 필요합니다 → 다음 장(03)의 주제입니다.

| | RDB (이번 장) | 벡터DB (다음 장) |
|---|---|---|
| 잘 맞는 질문 | "ρ·λ 가 가장 작은 3개", "Fe 첨가 시료 평균 결정립" | "Cu 저항이 커지는 이유", "결정립계 편석 효과" |
| 데이터 형태 | 표(정형 데이터) | 문장·문단(비정형 데이터) |
| 일치 방식 | 정확한 값·조건 | 의미가 비슷한 정도 |

In [ ]:
conn.close()   # DB 연결 닫기

## 정리

- **RDB** 는 데이터를 열 이름과 자료형이 정해진 **테이블** 에 저장하고, **기본키/외래키** 로 테이블을 연결합니다.
- SQL의 기본 문형: `SELECT ... FROM ... WHERE ... ORDER BY ... LIMIT`, 연결은 `JOIN`, 묶어 계산은 `GROUP BY`.
- 논문 표(Sung 2026 Table 1·2, Kim 2016 Table 1)를 DB로 만들면 "벌크 비저항은 Cu가 최저지만 ρ·λ 로는 Rh·Nb·Ir 등이 유리" 같은 결론을 **정확히** 확인할 수 있습니다.
- **Text2SQL** = 스키마 설명 + 질문 → LLM이 SQL 생성 → 실행. 반드시 **SELECT 전용 검사 + 읽기 전용 연결** 로 안전하게 실행하고, 생성된 SQL을 사람이 확인합니다.
- RDB는 숫자·조건에 강하지만, 논문 본문 같은 **비정형 텍스트의 의미 검색은 못 합니다** → 03장 벡터DB.

## 직접 해보기

1. `metals` 테이블에서 **fcc 구조** 금속만 골라 EMFP(`emfp_nm`)가 짧은 순으로 정렬하는 SQL을 직접 작성해 보세요.
2. `alloy_films` 에서 **Nb 첨가 시료** 의 평균 두께(`thickness_nm`)와 평균 결정립 크기를 `GROUP BY` 없이 `WHERE` 와 `AVG` 로 구해 보세요.
3. `text2sql()` 에 "응집 에너지가 가장 큰 화합물은?" 을 물어보고, 생성된 SQL이 정말 맞는지 직접 작성한 SQL 결과와 비교해 보세요. 틀렸다면 `PROMPT_TEMPLATE` 에 예시(질문–SQL 쌍)를 한 개 추가해 개선되는지 확인해 보세요.